# Insult/Desires Category Review

This notebook loads the evaluated comments from both samples, extracts comments classified as `Insult` or `Desires`, and calculates the percentage those categories represent within each sample.

## Load Data

In [9]:
from pathlib import Path
import html
import re

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", 220)

DATASETS = {
    "pseudoscientific": Path("Data/Comments/Evaluated_pseudo_multilingual.csv"),
    "scientific": Path("Data/Comments/Evaluated_scientific_multilingual.csv"),
}

def normalize_comment(value):
    value = "" if pd.isna(value) else str(value)
    value = html.unescape(value).strip()
    return re.sub(r"\s+", " ", value)

frames = []
for sample, path in DATASETS.items():
    df = pd.read_csv(path)
    df.insert(0, "sample", sample)
    frames.append(df)

comments = pd.concat(frames, ignore_index=True)
comments["comment_clean"] = comments["comment"].map(normalize_comment)
comments["etiqueta"] = comments["etiqueta"].fillna("").astype(str).str.strip()

sample_sizes = comments.groupby("sample").size().rename("sample_total")
display(sample_sizes.reset_index())

,sample,sample_total
0,pseudoscientific,32025
1,scientific,20386


## Extract Insult and Desires Comments

In [10]:
LABELS_TO_REVIEW = ["Insult", "Desires"]

review_columns = [
    "sample",
    "comment_id",
    "case_id",
    "conversation_id",
    "timestamp",
    "is_response",
    "response_of",
    "language",
    "emotion",
    "etiqueta",
    "comment",
    "comment_clean",
]
review_columns = [column for column in review_columns if column in comments.columns]

label_review_comments = (
    comments.loc[comments["etiqueta"].isin(LABELS_TO_REVIEW), review_columns]
    .sort_values(["sample", "etiqueta", "timestamp", "comment_id"])
    .reset_index(drop=True)
)

label_review_comments

,sample,comment_id,case_id,conversation_id,timestamp,is_response,response_of,language,emotion,etiqueta,comment,comment_clean
0,pseudoscientific,UgiFkq7l9FJOrngCoAEC,BlV8tEcu7Fk,UgiFkq7l9FJOrngCoAEC,2014-09-11 18:08:37+00:00,False,NaN,en,Neutral,Desires,I would like to give it a try....All I have to do is follow the video..\nisn't it?,I would like to give it a try....All I have to do is follow the video.. isn't it?
1,pseudoscientific,UghKaV-6XjmW8XgCoAEC.83-P6bUrVjX7-KfoMuFyVF,zV62tK-3F3w,UghKaV-6XjmW8XgCoAEC,2015-08-20 01:29:58+00:00,True,UghKaV-6XjmW8XgCoAEC,en,Positivo,Desires,"You are welcome. <br>Blessings, Lourdes","You are welcome. <br>Blessings, Lourdes"
2,pseudoscientific,Ugj38hBkboYUHXgCoAEC.87qzvzyksrJ87u8sKhiMHh,zV62tK-3F3w,Ugj38hBkboYUHXgCoAEC,2015-12-18 11:42:39+00:00,True,Ugj38hBkboYUHXgCoAEC,en,Positivo,Desires,"+Jack Ellul You are welcome. <br>Blessings, Lourdes","+Jack Ellul You are welcome. <br>Blessings, Lourdes"
3,pseudoscientific,UggZxY-MKHxaQHgCoAEC.8AL3cZcjLbo8AMKIBbgqYI,k7smbM1y55Y,UggZxY-MKHxaQHgCoAEC,2016-02-17 12:43:12+00:00,True,UggZxY-MKHxaQHgCoAEC,en,Positivo,Desires,"+Aka Dua You are welcome. <br>Blessings,<br>Lourdes","+Aka Dua You are welcome. <br>Blessings,<br>Lourdes"
4,pseudoscientific,UghyVUXtJWwLXHgCoAEC,Jk-0JC6giFI,UghyVUXtJWwLXHgCoAEC,2016-04-14 23:42:29+00:00,False,NaN,en,Positivo,Desires,Espetacular!!!,Espetacular!!!
...,...,...,...,...,...,...,...,...,...,...,...,...
3883,scientific,UgwREj1QhwTPwUrhPPJ4AaABAg.9sbXKeACEM6AHRJP1uKtgF,ScBzqEqRNAw,UgwREj1QhwTPwUrhPPJ4AaABAg,2025-04-27 21:49:49+00:00,True,UgwREj1QhwTPwUrhPPJ4AaABAg,es,Positivo,Insult,​@@sanchezsanchez6561 Así mismo es 🙏🏼,​@@sanchezsanchez6561 Así mismo es 🙏🏼
3884,scientific,Ugx-tcZoMyx-TXX7p3x4AaABAg,MlhR0xvrgwo,Ugx-tcZoMyx-TXX7p3x4AaABAg,2025-04-28 08:36:02+00:00,False,NaN,es,Neutral,Insult,Habla muuuucho enredo y naada claro respecto al titular!😊,Habla muuuucho enredo y naada claro respecto al titular!😊
3885,scientific,Ugy35usYT9rLme0hGIJ4AaABAg.AHW2C8Y5IAfAHXWrLbajqd,qcd5IYBSMEU,Ugy35usYT9rLme0hGIJ4AaABAg,2025-04-30 07:42:51+00:00,True,Ugy35usYT9rLme0hGIJ4AaABAg,hi,Positivo,Insult,👍🏻🤓,👍🏻🤓
3886,scientific,UgxZirqm2f7jo68gOkB4AaABAg.AHczNxu_Cz9AHd2PnPmael,V3vhXQy48jo,UgxZirqm2f7jo68gOkB4AaABAg,2025-05-02 20:31:30+00:00,True,UgxZirqm2f7jo68gOkB4AaABAg,en,Positivo,Insult,Wardshroomies is indeed him,Wardshroomies is indeed him


## Calculate Percentages

In [11]:
label_review_summary = (
    label_review_comments.groupby(["sample", "etiqueta"])
    .size()
    .rename("comments")
    .reset_index()
)
label_review_summary["sample_total"] = label_review_summary["sample"].map(sample_sizes)
label_review_summary["frequency_in_sample"] = label_review_summary["comments"] / label_review_summary["sample_total"]
label_review_summary["frequency_percent"] = label_review_summary["frequency_in_sample"] * 100

combined_target_summary = (
    label_review_comments.groupby("sample")
    .size()
    .rename("insult_desires_comments")
    .reset_index()
)
combined_target_summary["sample_total"] = combined_target_summary["sample"].map(sample_sizes)
combined_target_summary["frequency_in_sample"] = combined_target_summary["insult_desires_comments"] / combined_target_summary["sample_total"]
combined_target_summary["frequency_percent"] = combined_target_summary["frequency_in_sample"] * 100

display(label_review_summary)
display(combined_target_summary)

,sample,etiqueta,comments,sample_total,frequency_in_sample,frequency_percent
0,pseudoscientific,Desires,1782,32025,0.055644,5.564403
1,pseudoscientific,Insult,1320,32025,0.041218,4.121780
2,scientific,Desires,289,20386,0.014176,1.417640
3,scientific,Insult,497,20386,0.024379,2.437948


,sample,insult_desires_comments,sample_total,frequency_in_sample,frequency_percent
0,pseudoscientific,3102,32025,0.096862,9.686183
1,scientific,786,20386,0.038556,3.855587
